<a href="https://colab.research.google.com/github/ScG4m3rLOL/Modelos_1_Udea_2025_2_Proyecto_Kaggle/blob/main/03-modelo%20soluci%C3%B3n%20con%20DesicionTreeClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Competencia Kaggle

#Modelos y Simulación I Udea2025-2

In [1]:
!wget --no-cache -O init.py -q https://raw.githubusercontent.com/rramosp/ai4eng.v1/main/content/init.py
import init; init.init(force_download=False); init.get_weblink()

replicating local resources


Configuración de Kaggle y descarga de datos

In [6]:
!mv kaggle.json /root/.config/kaggle/kaggle.json
!chmod 600 /root/.config/kaggle/kaggle.json

!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia
!unzip udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip
!unzip -l udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip

mv: cannot stat 'kaggle.json': No such file or directory
udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip
  inflating: submission_example.csv  
  inflating: test.csv                
  inflating: train.csv               
Archive:  udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
  4716673  2025-09-16 01:46   submission_example.csv
 59185238  2025-09-16 01:46   test.csv
143732437  2025-09-16 01:46   train.csv
---------                     -------
207634348                     3 files


Carga de librerías

In [23]:
import pandas as pd
import numpy as np
import unicodedata
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

Carga de datasets

In [35]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")

Train: (692500, 21)
Test: (296786, 20)


Identificar variable objetivo

In [36]:
target_candidates = list(set(train.columns) - set(test.columns))
variable_objetivo = target_candidates[0]
print(f"Variable objetivo: {variable_objetivo}")

Variable objetivo: RENDIMIENTO_GLOBAL


Preprocesamiento

In [37]:
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    return texto

columnas_comunes = list(set(train.columns) & set(test.columns))
columnas_categoricas = [col for col in columnas_comunes if train[col].dtype == 'object']

for df in [train, test]:
    for col in columnas_categoricas:
        df[f"{col}_numerico"] = df[col].apply(normalizar_texto).astype('category').cat.codes

In [38]:
X_train = train.drop(columns=[variable_objetivo])
y_train = train[variable_objetivo]

columnas_numericas = [col for col in X_train.columns
                     if col in test.columns or col.endswith('_numerico')]
columnas_numericas = [col for col in columnas_numericas
                     if X_train[col].dtype in [np.number] and col != variable_objetivo]

print(f"Columnas para modelo: {len(columnas_numericas)}")

Columnas para modelo: 4


Entrenamiento y evaluación

In [39]:
X_ent, X_val, y_ent, y_val = train_test_split(X_train[columnas_numericas], y_train, test_size=0.2, random_state=42)

modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_ent, y_ent)

y_pred_val = modelo.predict(X_val)
print("Accuracy validación:", accuracy_score(y_val, y_pred_val))

Accuracy validación: 0.2687148014440433


Predicción final

In [40]:
modelo_final = DecisionTreeClassifier(random_state=42)
modelo_final.fit(X_train[columnas_numericas], y_train)

predicciones = modelo_final.predict(test[columnas_numericas])

id_column = test.columns[0]
submission = pd.DataFrame({
    id_column: test[id_column],
    variable_objetivo: predicciones
})

In [41]:
submission.to_csv('submission.csv', index=False)
from google.colab import files
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>